In [0]:
!pip install fastexcel

In [0]:
import pandas as pd
import polars as pl
import fastexcel
import pickle
import numpy as np
import os
import warnings
import pyarrow as pa
import pyarrow.parquet as pq
import calendar
from datetime import date
warnings.filterwarnings('ignore')
spark.conf.set("spark.sql.adaptive.enabled", "false") 

In [0]:
ipcap_path = '/Volumes/opsanalytics_adb_workspace01/or/ipcap_inputs'
output_path = '/Volumes/opsanalytics_adb_workspace01/or/model_outputs'
or_case_table = 'opsanalytics_adb_workspace01.or.or_quality_dashboard_case_details'
holidays_table = 'opsanalytics_adb_workspace01.lab.mshs_holidays'


simulation_table_names = ['AGGXCON_simulation_1_2026',
 'AGGXCON_simulation_1_2027',
 'AGGXCON_simulation_1_2028',
 'AGGXCON_simulation_1_2029',
 'AGGXCON_simulation_1_2030',
 'AGGXCON_simulation_2_2026',
 'AGGXCON_simulation_2_2027',
 'AGGXCON_simulation_2_2028',
 'AGGXCON_simulation_2_2029',
 'AGGXCON_simulation_2_2030',
 'AGGXCON_simulation_3_2026',
 'AGGXCON_simulation_3_2027',
 'AGGXCON_simulation_3_2028',
 'AGGXCON_simulation_3_2029',
 'AGGXCON_simulation_3_2030',
 'AGGXCON_simulation_4_2026',
 'AGGXCON_simulation_4_2027',
 'AGGXCON_simulation_4_2028',
 'AGGXCON_simulation_4_2029',
 'AGGXCON_simulation_4_2030',
 'AGGXCON_simulation_5_2026',
 'AGGXCON_simulation_5_2027',
 'AGGXCON_simulation_5_2028',
 'AGGXCON_simulation_5_2029',
 'AGGXCON_simulation_5_2030',
 'AGGxAGG_simulation_1_2026',
 'AGGxAGG_simulation_1_2027',
 'AGGxAGG_simulation_1_2028',
 'AGGxAGG_simulation_1_2029',
 'AGGxAGG_simulation_1_2030',
 'AGGxAGG_simulation_2_2026',
 'AGGxAGG_simulation_2_2027',
 'AGGxAGG_simulation_2_2028',
 'AGGxAGG_simulation_2_2029',
 'AGGxAGG_simulation_2_2030',
 'AGGxAGG_simulation_3_2026',
 'AGGxAGG_simulation_3_2027',
 'AGGxAGG_simulation_3_2028',
 'AGGxAGG_simulation_3_2029',
 'AGGxAGG_simulation_3_2030',
 'AGGxAGG_simulation_4_2026',
 'AGGxAGG_simulation_4_2027',
 'AGGxAGG_simulation_4_2028',
 'AGGxAGG_simulation_4_2029',
 'AGGxAGG_simulation_4_2030',
 'AGGxAGG_simulation_5_2026',
 'AGGxAGG_simulation_5_2027',
 'AGGxAGG_simulation_5_2028',
 'AGGxAGG_simulation_5_2029',
 'AGGxAGG_simulation_5_2030',
 'CONXAGG_simulation_1_2026',
 'CONXAGG_simulation_1_2027',
 'CONXAGG_simulation_1_2028',
 'CONXAGG_simulation_1_2029',
 'CONXAGG_simulation_1_2030',
 'CONXAGG_simulation_2_2026',
 'CONXAGG_simulation_2_2027',
 'CONXAGG_simulation_2_2028',
 'CONXAGG_simulation_2_2029',
 'CONXAGG_simulation_2_2030',
 'CONXAGG_simulation_3_2026',
 'CONXAGG_simulation_3_2027',
 'CONXAGG_simulation_3_2028',
 'CONXAGG_simulation_3_2029',
 'CONXAGG_simulation_3_2030',
 'CONXAGG_simulation_4_2026',
 'CONXAGG_simulation_4_2027',
 'CONXAGG_simulation_4_2028',
 'CONXAGG_simulation_4_2029',
 'CONXAGG_simulation_4_2030',
 'CONXAGG_simulation_5_2026',
 'CONXAGG_simulation_5_2027',
 'CONXAGG_simulation_5_2028',
 'CONXAGG_simulation_5_2029',
 'CONXAGG_simulation_5_2030',
 'CONxCON_simulation_1_2026',
 'CONxCON_simulation_1_2027',
 'CONxCON_simulation_1_2028',
 'CONxCON_simulation_1_2029',
 'CONxCON_simulation_1_2030',
 'CONxCON_simulation_2_2026',
 'CONxCON_simulation_2_2027',
 'CONxCON_simulation_2_2028',
 'CONxCON_simulation_2_2029',
 'CONxCON_simulation_2_2030',
 'CONxCON_simulation_3_2026',
 'CONxCON_simulation_3_2027',
 'CONxCON_simulation_3_2028',
 'CONxCON_simulation_3_2029',
 'CONxCON_simulation_3_2030',
 'CONxCON_simulation_4_2026',
 'CONxCON_simulation_4_2027',
 'CONxCON_simulation_4_2028',
 'CONxCON_simulation_4_2029',
 'CONxCON_simulation_4_2030',
 'CONxCON_simulation_5_2026',
 'CONxCON_simulation_5_2027',
 'CONxCON_simulation_5_2028',
 'CONxCON_simulation_5_2029',
 'CONxCON_simulation_5_2030']

In [0]:
from pyspark.sql import functions as F

# 1. Load the OR Case Table natively (replaces the SQL and .toPandas())
df_spark  = spark.table(or_case_table)
df = df_spark.pandas_api()


# 2. Load the Holidays Table natively 
holidays_df = spark.table(holidays_table)
holidays_df = holidays_df.pandas_api()
holidays_df['holiday_date'] = holidays_df['holiday_date'].astype('datetime64[ns]')

In [0]:
df_25 = df.loc[df['PATIENT_IN_ROOM_DTTM'].notnull()]
df_25 = df.loc[(df['SURGERY_DATE'] >= '2025-01-01') & (df['SURGERY_DATE'] <= '2025-12-31')]
# Exclude Weekends
df_25 = df_25.loc[~df_25['SURGERY_DATE'].dt.dayofweek.isin([5,6])]
# Exclude Holidays
df_25 = df_25.loc[~df_25['SURGERY_DATE'].isin(holidays_df['holiday_date'].to_list())]
df_25['case_minutes'] = (df_25['PATIENT_OUT_ROOM_DTTM'] - df_25['PATIENT_IN_ROOM_DTTM']) / 60


total_case_minutes = df_25.groupby("HOSPITAL")[['case_minutes']].sum().reset_index()
# total_case_minutes['scenario'] = 'historical'
total_case_minutes['year'] = 2025

total_case_minutes = total_case_minutes.to_pandas()
total_case_minutes = total_case_minutes.rename(
    columns ={
        'case_minutes': '2025',}
)
total_case_minutes = total_case_minutes[['HOSPITAL','2025']]
display(total_case_minutes)

In [0]:
def join_scenario_and_or_data(scenario,exclude_holidays_weekends=True):

    query = f""" 
    with dedup as (
      select distinct ENCOUNTER_NO, MSMRN, ADMIT_DT_SRC, DSCH_DT_SRC, NEW_ADMIT_DT_SRC ,NEW_DSCH_DT_SRC, SIMULATION, PROJECTION_YEAR
      from opsanalytics_adb_workspace01.or.{scenario}
      WHERE MSMRN LIKE '%\_%')
    select 
        a.HOSPITAL, 
        a.PATIENT_IN_ROOM_DTTM, 
        a.PATIENT_OUT_ROOM_DTTM, 
        a.SURGERY_DATE, 
        a.SURGEON_SPECIALTY, 
        a.OR_ID, 
        a.OR_LOCATION, 
        a.CASE_STATUS, 
        a.OR_CASE_ID, 
        b.ENCOUNTER_NO, 
        b.MSMRN, 
        element_at(split(b.MSMRN, '_'), 1) as PARENT_MRN,  
        b.ADMIT_DT_SRC, 
        b.DSCH_DT_SRC, 
        b.NEW_ADMIT_DT_SRC, 
        b.NEW_DSCH_DT_SRC 
    from opsanalytics_adb_workspace01.or.or_quality_dashboard_case_details a 
    join dedup b 
      on a.PAT_MRN_ID = element_at(split(b.MSMRN, '_'), 1) 
      and a.SURGERY_DATE between b.admit_dt_src and b.dsch_dt_src 
    where a.patient_in_room_dttm is not null 
    """opsanalytics_adb_workspace01.or.aggxagg_simulation_1_2026
    case,sim,sim_run,year=scenario.split("_")

    df = spark.sql(query)
    df = df.toPandas()

    if exclude_holidays_weekends:
      # Exclude Weekends
      df = df.loc[~df['SURGERY_DATE'].dt.dayofweek.isin([5,6])]
      # Exclude Holidays
      df = df.loc[~df['SURGERY_DATE'].isin(holidays_df['holiday_date'].to_list())]

    #df['actual_los'] = df['DSCH_DT_SRC'] - df['ADMIT_DT_SRC']
    #df['actual_los'] = df['actual_los'].dt.days
    #df['new_los'] = df['NEW_DSCH_DT_SRC'] - df['NEW_ADMIT_DT_SRC']
    #df['new_los'] = df['new_los'].dt.days
    #df['surgery_offset'] = df['SURGERY_DATE'] - df['ADMIT_DT_SRC']
    #df['surgery_offset'] = df['surgery_offset'].dt.days
    df['scenario'] = case
    df['simulation_run'] = sim_run
    df['simulation_year'] = year


    #df['NEW_SURGERY_DATE'] = df['NEW_ADMIT_DT_SRC'] + pd.to_timedelta(df['surgery_offset'], unit='D')
    #df['NEW_SURGERY_YEAR'] = df['NEW_SURGERY_DATE'].dt.year
    #date_shift = df['NEW_SURGERY_DATE'] - df['SURGERY_DATE']
    #df['NEW_SURGERY_DATE'] = adjust_surgery_dates(df, holidays_df)
    #df['NEW_PATIENT_IN_ROOM_DTTM'] = df['PATIENT_IN_ROOM_DTTM'] + date_shift
    #df['NEW_PATIENT_OUT_ROOM_DTTM'] = df['PATIENT_OUT_ROOM_DTTM'] + date_shift
    df['case_minutes'] = df['PATIENT_OUT_ROOM_DTTM'] - df['PATIENT_IN_ROOM_DTTM']
    df['case_minutes'] = df['case_minutes'].dt.total_seconds()/60
    print(df.shape)

    df_agg = df.groupby(['HOSPITAL', 'scenario', 'simulation_year', 'simulation_run']).agg(
        case_minutes=('case_minutes', 'sum')).reset_index()


    return df_agg

In [0]:
join_scenario_and_or_data('aggxagg_simulation_1_2026')

In [0]:
def get_case_minutes_simulation(simulation_data):
    # 1. Filter out MSBI early to free up memory before aggregating
    filtered_data = simulation_data[simulation_data['HOSPITAL'] != 'MSBI']
    
    # 2. Group by and aggregate both metrics in a single pass
    total_case_minutes = (
        filtered_data.groupby(["HOSPITAL", "scenario", "simulation_run", "NEW_SURGERY_YEAR"])
        .agg(
            case_minutes=('case_minutes', 'sum')        )
        .reset_index()
    )
    total_case_minutes.rename(columns={'NEW_SURGERY_YEAR': 'year'}, inplace=True)
    # total_case_minutes['surgery_days'] = 365 + calendar.isleap(total_case_minutes['year'])

    # 3. Calculate the average minutes directly
    # total_case_minutes['avg_minutes'] = total_case_minutes['case_minutes'] / total_case_minutes['NEW_SURGERY_DATE']
    
    return total_case_minutes

In [0]:
simualtion_or_cases = {scenario : join_scenario_and_or_data(scenario,exclude_holidays_weekends=True) for scenario in simulation_table_names}

In [0]:
simualtion_or_cases_hw_off = {scenario : join_scenario_and_or_data(scenario,exclude_holidays_weekends=False) for scenario in simulation_table_names}

In [0]:
simualtion_or_cases['AGGXCON_simulation_2_2026'].shape[0]

In [0]:
simualtion_or_cases_hw_off['AGGXCON_simulation_2_2026'].shape[0]

In [0]:
simualtion_or_cases['AGGXCON_simulation_2_2027'].shape[0]

In [0]:
simualtion_or_cases_hw_off['AGGXCON_simulation_2_2027'].shape[0]

In [0]:
simulation_case_minutes = {scenario : get_case_minutes_simulation(simualtion_or_cases[scenario]) for scenario in simualtion_or_cases}
simulation_case_minutes_hw_off = {scenario : get_case_minutes_simulation(simualtion_or_cases_hw_off[scenario]) for scenario in simualtion_or_cases}

In [0]:
simulation_case_minutes_combined_df = pd.concat(simulation_case_minutes,
                                            ignore_index=True)
simulation_case_minutes_combined_df = (
    simulation_case_minutes_combined_df.groupby(["HOSPITAL", "scenario", "simulation_run", "year"])
    .agg(case_minutes=('case_minutes', 'sum'))
    .reset_index()
    )
# Aggregate Across all Simulations
simulation_case_minutes_combined_df = (
    simulation_case_minutes_combined_df.groupby(["HOSPITAL", "scenario", "year"])
    .agg(case_minutes=('case_minutes', 'mean'))
    .reset_index()
    )
# simulation_case_minutes_combined_df['working_days'] = [
#     int(np.busday_count(
#         f"{int(y)}-01-01", 
#         f"{int(y) + 1}-01-01", 
#         # Safely convert the resulting array to datetime64[D] afterwards
#         holidays=holidays_df.loc[holidays_df['holiday_date'].dt.year == int(y), 'holiday_date'].to_numpy().astype('datetime64[D]')
#     )) 
#     for y in simulation_case_minutes_combined_df['year'].to_numpy()
# ]

# ---------------------------- HW OFF -----------------------------------
simulation_case_minutes_combined_df_hw_off = pd.concat(simulation_case_minutes_hw_off,
                                            ignore_index=True)
simulation_case_minutes_combined_df_hw_off = (
    simulation_case_minutes_combined_df_hw_off.groupby(["HOSPITAL", "scenario", "simulation_run", "year"])
    .agg(case_minutes=('case_minutes', 'sum'))
    .reset_index()
    )
# Aggregate Across all Simulations
simulation_case_minutes_combined_df_hw_off = (
    simulation_case_minutes_combined_df_hw_off.groupby(["HOSPITAL", "scenario", "year"])
    .agg(case_minutes=('case_minutes', 'mean'))
    .reset_index()
    )
# simulation_case_minutes_combined_df['working_days'] = [
#     int(np.busday_count(
#         f"{int(y)}-01-01", 
#         f"{int(y) + 1}-01-01", 
#         # Safely convert the resulting array to datetime64[D] afterwards
#         holidays=holidays_df.loc[holidays_df['holiday_date'].dt.year == int(y), 'holiday_date'].to_numpy().astype('datetime64[D]')
#     )) 
#     for y in simulation_case_minutes_combined_df['year'].to_numpy()
# ]

In [0]:
simulation_case_minutes_combined_df_nw = simulation_case_minutes_combined_df[['HOSPITAL','scenario','year','case_minutes']]
simulation_case_minutes_combined_df_nw_hw_off = simulation_case_minutes_combined_df_hw_off[['HOSPITAL','scenario','year','case_minutes']]

In [0]:
simulation_case_minutes_excel = simulation_case_minutes_combined_df_nw.pivot_table(index=['HOSPITAL','scenario'],
                                    columns='year',
                                    values=['case_minutes'])

# 2. Drop the top level ('Sales') so only the years ('2025', '2026') remain
simulation_case_minutes_excel.columns = simulation_case_minutes_excel.columns.droplevel(0)

# 3. Delete the column index name metadata
simulation_case_minutes_excel.columns.name = None

# 4. Bring your index back as regular columns
simulation_case_minutes_excel = simulation_case_minutes_excel.reset_index()

# === FIX: Ensure HOSPITAL is the index of total_case_minutes ===
if 'HOSPITAL' in total_case_minutes.columns:
    total_case_minutes = total_case_minutes.set_index('HOSPITAL')

# === FIX: Align data types to avoid the previous int64/object error ===
simulation_case_minutes_excel['HOSPITAL'] = simulation_case_minutes_excel['HOSPITAL'].astype(str)
total_case_minutes.index = total_case_minutes.index.astype(str)
# ==============================================================

# 5. Join your data seamlessly
simulation_case_minutes_excel = simulation_case_minutes_excel.join(total_case_minutes, on='HOSPITAL', how='left')
simulation_case_minutes_excel =  simulation_case_minutes_excel[['HOSPITAL',
                                                                'scenario',
                                                                '2025',2026,2027,2028,2029,2030,2031]]
simulation_case_minutes_excel.to_csv(output_path+'/OR_DELTAS_AFTER_FUNCTION_IMPLEMENTATION.csv',index=False)

In [0]:
simulation_case_minutes_excel_hw_off = simulation_case_minutes_combined_df_nw_hw_off.pivot_table(index=['HOSPITAL','scenario'],
                                    columns='year',
                                    values=['case_minutes'])

# 2. Drop the top level ('Sales') so only the years ('2025', '2026') remain
simulation_case_minutes_excel_hw_off.columns = simulation_case_minutes_excel_hw_off.columns.droplevel(0)

# 3. Delete the column index name metadata
simulation_case_minutes_excel_hw_off.columns.name = None

# 4. Bring your index back as regular columns
simulation_case_minutes_excel_hw_off = simulation_case_minutes_excel_hw_off.reset_index()

# === FIX: Ensure HOSPITAL is the index of total_case_minutes ===
if 'HOSPITAL' in total_case_minutes.columns:
    total_case_minutes = total_case_minutes.set_index('HOSPITAL')

# === FIX: Align data types to avoid the previous int64/object error ===
simulation_case_minutes_excel_hw_off['HOSPITAL'] = simulation_case_minutes_excel_hw_off['HOSPITAL'].astype(str)
total_case_minutes.index = total_case_minutes.index.astype(str)
# ==============================================================

# 5. Join your data seamlessly
simulation_case_minutes_excel_hw_off = simulation_case_minutes_excel_hw_off.join(total_case_minutes, on='HOSPITAL', how='left')
simulation_case_minutes_excel_hw_off =  simulation_case_minutes_excel_hw_off[['HOSPITAL',
                                                                'scenario',
                                                                '2025',2026,2027,2028,2029,2030,2031]]
simulation_case_minutes_excel_hw_off.to_csv(output_path+'/OR_DELTAS_HW_OFF_AFTER_FUNCTION_IMPLEMENTATION.csv',index=False)

In [0]:
display(simulation_case_minutes_excel)

In [0]:
display(simulation_case_minutes_excel_hw_off)

In [0]:
# Keep the first two columns, accumulate from the third column onward
# df_growth = simulation_case_minutes_excel.copy()
# df_growth.iloc[:, 2:] = df_growth.iloc[:, 2:].cumsum(axis=1)

In [0]:
# simualtion_or_cases_combined_df.loc[simualtion_or_cases_combined_df['PARENT_MRN']!=simualtion_or_cases_combined_df['MSMRN']][['case_minutes',
#                                                                                                                               'PARENT_MRN','MSMRN',
#                                                                                                                               'PATIENT_OUT_ROOM_DTTM',
#                                                                                                                               'PATIENT_IN_ROOM_DTTM',
#                                                                                                                               'SURGERY_DATE', 'NEW_SURGERY_DATE']]

## Working Area